In [1]:
import os
import numpy as np
import librosa
from tqdm import tqdm

In [2]:
data_path = "../data"

In [3]:
labels = sorted(os.listdir(data_path))

print(labels[:20])
print("Total classes:", len(labels))

['.DS_Store', 'LICENSE', 'README.md', '_background_noise_', 'backward', 'bed', 'bird', 'cat', 'dog', 'down', 'eight', 'five', 'follow', 'forward', 'four', 'go', 'happy', 'house', 'learn', 'left']
Total classes: 42


In [4]:
labels = [
    l for l in os.listdir(data_path)
    if os.path.isdir(os.path.join(data_path,l))
    and l != "_background_noise_"
]
print("Classes:", labels)
print("Total classes:", len(labels))

Classes: ['up', 'bird', 'off', 'nine', 'house', 'down', 'bed', 'go', 'happy', 'six', 'five', 'no', 'cat', 'on', 'seven', 'three', 'wow', 'visual', 'follow', 'yes', 'right', 'marvin', 'eight', 'sheila', 'one', 'zero', 'forward', 'learn', 'four', 'backward', 'tree', 'two', 'left', 'dog', 'stop']
Total classes: 35


In [5]:
def add_noise(audio):
    
    noise = np.random.randn(len(audio))
    
    audio = audio + 0.005 * noise
    
    return audio


def shift_audio(audio):
    
    shift = np.random.randint(1600)
    
    return np.roll(audio, shift)

In [6]:
def extract_mfcc(file_path):

    audio, sr = librosa.load(file_path, sr=16000)

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=40
    )

    return mfcc

In [7]:
max_len = 44

def pad_mfcc(mfcc):

    if mfcc.shape[1] < max_len:
        pad_width = max_len - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0,0),(0,pad_width)), mode='constant')

    else:
        mfcc = mfcc[:, :max_len]

    return mfcc

In [8]:
X = []
y = []

for label in tqdm(labels):

    folder = os.path.join(data_path, label)

    for file in os.listdir(folder):

        if not file.endswith(".wav"):
            continue

        file_path = os.path.join(folder, file)

        mfcc = extract_mfcc(file_path)

        mfcc = pad_mfcc(mfcc)   

        X.append(mfcc)
        y.append(label)

100%|██████████| 35/35 [10:26<00:00, 17.89s/it]


In [9]:
X = np.array(X)
y = np.array(y)

print("Feature shape:", X.shape)
print("Labels shape:", y.shape)

Feature shape: (105829, 40, 44)
Labels shape: (105829,)


In [10]:
X = X[..., np.newaxis]

print(X.shape)

(105829, 40, 44, 1)


In [11]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

print(y_encoded[:10])

[30 30 30 30 30 30 30 30 30 30]


In [12]:
np.save("../src/X_cnn_features.npy", X)
np.save("../src/y_cnn_labels.npy", y_encoded)